# Public Notice Check (V9)
**A last check before a bilingual notice goes out.**
Veritas AI International High School Hackathon 2026 · Track: AI + Languages

A programme coordinator pastes the Traditional Chinese and English versions of a notice.
The tool flags sentences where the two versions may tell families different things
(when to apply, who qualifies, what to do), and says why. A person always makes the final call.

**How V9 checks a notice, in three layers**
1. **Clause facts (V9):** each language is read on its own against hand-written clause patterns
   (fees, refunds, age limits, who pays postage, where a signed form goes, ...). Facts found in both
   versions are compared value by value, quoting the exact words in each language.
2. **Evidence rules (V8 and earlier):** numbers, clock times, negation, deadline vs event date,
   weekdays, actions, web links. Each rule stays silent when it cannot read a feature on both sides.
3. **Pretrained models:** LaBSE matches sentences across languages; mDeBERTa NLI scores whether two
   sentences contradict each other.

**What is pretrained and what is ours**
- Pretrained (not trained by us): **LaBSE** and **mDeBERTa NLI**. We use the weights as released.
- Built by us: every rule and clause pattern, the logic that combines them, the test data and the evaluation.

**How to run:** Runtime → Change runtime type → T4 GPU (optional, faster), then Runtime → Run all.
The first run downloads the two models (about 2.5 GB). Data files are fetched from our GitHub repository
if they have not been uploaded.

| Section | What it shows |
|---|---|
| 7 | Development results on our own sentence pairs |
| 8 | Real government notices (the blind scores we report) |
| 11 | V8 vs V9 on 1,000 synthetic development cases |
| 12 | **V9 on 1,000 held-out synthetic cases, scored once** |

## 1. Setup

In [ ]:
!pip -q install sentence-transformers==5.7.0 transformers==5.22.1 gradio sentencepiece

import re
import itertools
import os
import urllib.request
import pandas as pd
import numpy as np
import torch
torch.manual_seed(8)

DATA_URL = "https://raw.githubusercontent.com/SISHK-AI-Club/Veritas-AI-Hackathon/main/data/"
def get_data(name):
    # use an uploaded copy if there is one, otherwise download it from our GitHub repository
    if not os.path.exists(name):
        try:
            urllib.request.urlretrieve(DATA_URL + name, name)
        except Exception as exc:
            print(f"Could not download {name}: {exc}")
    return os.path.exists(name)

for name in ["notices.csv", "fresh_test.csv", "fresh_test_2.csv", "v9_development_1000.csv", "v9_unseen_1000.csv"]:
    print(name, "ready" if get_data(name) else "missing")

## 2. Data
Each row is one **sentence pair**.
- `has_error` = 1 if the English is wrong or misleading, 0 if it is a correct translation
- `error_type` = `none`, `number`, `negation`, `meaning`, `date_role` or `awkward`
- `notice_id` groups the correct and broken versions of the same sentence, so they always land in the same split

**Data:** if a file called `notices.csv` has been uploaded (folder icon on the left), it is used. Otherwise the short built-in list below is used.

In [ ]:
rows = [
  # notice_id, chinese, english, has_error, error_type
  (1, "報名截止日期為2026年12月5日。", "The application deadline is 5 December 2026.", 0, "none"),
  (1, "報名截止日期為2026年12月5日。", "The application deadline is 15 December 2026.", 1, "number"),
  (2, "本活動只限65歲或以上長者參加。", "This activity is only open to elderly people aged 65 or above.", 0, "none"),
  (2, "本活動只限65歲或以上長者參加。", "This activity is open to everyone aged 65 or below.", 1, "meaning"),
  (3, "參加者毋須繳付任何費用。", "Participants do not need to pay any fee.", 0, "none"),
  (3, "參加者毋須繳付任何費用。", "Participants need to pay a fee.", 1, "negation"),
  (4, "報名截止日期為11月20日，活動將於11月27日舉行。", "The application deadline is 20 November, and the activity will be held on 27 November.", 0, "none"),
  (4, "報名截止日期為11月20日，活動將於11月27日舉行。", "The application deadline is 27 November, and the activity will be held on 20 November.", 1, "date_role"),
  (5, "請於11月20日下午5時前交回表格。", "Please return the form by 5 p.m. on 20 November.", 0, "none"),
  (5, "請於11月20日下午5時前交回表格。", "Please return the form by 5 p.m. on 30 November.", 1, "number"),
  (6, "如遇八號或以上颱風信號，活動將會取消。", "If Typhoon Signal No. 8 or above is hoisted, the activity will be cancelled.", 0, "none"),
  (6, "如遇八號或以上颱風信號，活動將會取消。", "If Typhoon Signal No. 8 or above is hoisted, the activity will still go ahead.", 1, "meaning"),
  (7, "名額有限，先到先得。", "Places are limited and will be allocated on a first-come, first-served basis.", 0, "none"),
  (7, "名額有限，先到先得。", "The quota is limited, first arrive first get.", 1, "awkward"),
  (8, "學生必須由家長陪同出席。", "Students must be accompanied by a parent.", 0, "none"),
  (8, "學生必須由家長陪同出席。", "Students may attend without a parent.", 1, "negation"),
  (9, "查詢請致電2345 6789。", "For enquiries, please call 2345 6789.", 0, "none"),
  (9, "查詢請致電2345 6789。", "For enquiries, please call 2345 6798.", 1, "number"),
  (10, "參加者必須於活動當日帶備身份證明文件。", "Participants must bring an identity document on the day of the activity.", 0, "none"),
  (10, "參加者必須於活動當日帶備身份證明文件。", "Participants may bring an identity document on the day of the activity.", 1, "meaning"),
  (11, "只限香港居民申請。", "Only Hong Kong residents may apply.", 0, "none"),
  (11, "只限香港居民申請。", "Applications are open to everyone, including visitors.", 1, "meaning"),
  (12, "活動將於2026年12月12日（星期六）舉行。", "The activity will be held on Saturday, December 12, 2026.", 0, "none"),
  (12, "活動將於2026年12月12日（星期六）舉行。", "The activity will be held on Sunday, December 13, 2026.", 1, "number"),
  (13, "申請截止日期為12月1日，課程將於12月8日開始。", "Applications close on 1 December, and the course starts on 8 December.", 0, "none"),
  (13, "申請截止日期為12月1日，課程將於12月8日開始。", "Applications close on 8 December, and the course starts on 1 December.", 1, "date_role"),
]
df = pd.DataFrame(rows, columns=["notice_id", "chinese", "english", "has_error", "error_type"])

import os
if os.path.exists("notices.csv"):          # upload notices.csv with the folder icon to use the full starter set
    df = pd.read_csv("notices.csv")

# Split by notice: about 1/3 of notices for tuning (dev), the rest for the final score (test).
# Nothing is ever tuned on test rows.
ids = sorted(df["notice_id"].unique())
dev_ids = set(ids[: max(1, len(ids) // 3)])
df["split"] = df["notice_id"].apply(lambda i: "dev" if i in dev_ids else "test")
print(df["split"].value_counts())
df

## 3. Simplest baseline: a length rule
Flag a pair if the English is unusually long or short compared with the Chinese.
The allowed range is learned from the **correct** pairs in the dev split.

In [ ]:
def length_ratio(zh, en):
    return len(en.split()) / max(1, len(re.sub(r"\s", "", zh)))

dev_ok = df[(df.split == "dev") & (df.has_error == 0)]
ratios = [length_ratio(z, e) for z, e in zip(dev_ok.chinese, dev_ok.english)]
LOW, HIGH = min(ratios) * 0.8, max(ratios) * 1.2
print(f"Allowed English-words-per-Chinese-character range: {LOW:.2f} to {HIGH:.2f}")

def baseline_flag(zh, en):
    r = length_ratio(zh, en)
    return (r < LOW or r > HIGH), (["length looks off"] if (r < LOW or r > HIGH) else [])

## 4. The pretrained models
- **LaBSE** turns a sentence in any language into a vector. Similar meaning → similar vectors. We use it to match sentences and to spot pairs that are about different things.
- **mDeBERTa NLI** reads two sentences and says whether the second one *follows from*, is *unrelated to*, or **contradicts** the first. We ask it in both directions (Chinese → English and English → Chinese) and keep the higher contradiction score.

In [ ]:
from sentence_transformers import SentenceTransformer, util
from transformers import AutoTokenizer, AutoModelForSequenceClassification

device = "cuda" if torch.cuda.is_available() else "cpu"
labse = SentenceTransformer("sentence-transformers/LaBSE", device=device)

NLI_NAME = "MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7"
nli_tok = AutoTokenizer.from_pretrained(NLI_NAME)
nli_model = AutoModelForSequenceClassification.from_pretrained(NLI_NAME).to(device).eval()
# find which output position means "contradiction" from the model's own label list
CONTRA = [i for i, name in nli_model.config.id2label.items() if "contra" in name.lower()][0]
print("NLI labels:", nli_model.config.id2label)

def similarity(zh, en):
    a, b = labse.encode([zh, en], convert_to_tensor=True, normalize_embeddings=True)
    return float(util.cos_sim(a, b))

def contradiction(zh, en):
    # probability that the two sentences contradict each other (higher of the two directions)
    scores = []
    for premise, hypothesis in [(zh, en), (en, zh)]:
        inputs = nli_tok(premise, hypothesis, truncation=True, max_length=256, return_tensors="pt").to(device)
        with torch.no_grad():
            probs = torch.softmax(nli_model(**inputs).logits[0], -1)
        scores.append(float(probs[CONTRA]))
    return max(scores)

## 5. Our rules
1. **Numbers:** every number in one version should appear in the other. Handles Chinese numerals (八 → 8) and English month names (December → 12).
2. **Negation:** if one version says "not / no / without" and the other doesn't, that's a red flag.
3. **Date roles (new):** a date is labelled as an **application deadline** or an **event date** from the words around it.
   If both versions contain the same dates but the *deadline* in one is the *event date* in the other, the plain number check sees nothing wrong. This check catches it.

In [ ]:
CN_DIGITS = {"零": 0, "〇": 0, "一": 1, "二": 2, "兩": 2, "三": 3, "四": 4, "五": 5, "六": 6, "七": 7, "八": 8, "九": 9}
CN_UNITS = {"十": 10, "百": 100, "千": 1000}
CN_NUM = "零〇一二兩三四五六七八九十百千"

def cn_to_int(s):
    if s.isdigit():
        return int(s)
    total, num = 0, 0
    for ch in s:
        if ch in CN_DIGITS:
            num = CN_DIGITS[ch]
        elif ch in CN_UNITS:
            total += (num or 1) * CN_UNITS[ch]
            num = 0
    return total + num

MONTH_NAMES = ["january", "february", "march", "april", "may", "june", "july",
               "august", "september", "october", "november", "december"]
MONTHS = {m: i + 1 for i, m in enumerate(MONTH_NAMES)}
EN_WORDS = {"one": 1, "two": 2, "three": 3, "four": 4, "five": 5, "six": 6, "seven": 7, "eight": 8,
            "nine": 9, "ten": 10, "eleven": 11, "twelve": 12}

def numbers_zh(text):
    text = re.sub(r"(星期|週|周|禮拜)[一二三四五六日天]", "", text)   # weekdays are not numbers
    nums = {int(n) for n in re.findall(r"\d+", text)}
    nums |= {cn_to_int(m) for m in re.findall(rf"[{CN_NUM}]+", text)}
    return nums

def numbers_en(text):
    t = text.lower()
    nums = {int(n) for n in re.findall(r"\d+", t)}
    nums |= {v for k, v in {**MONTHS, **EN_WORDS}.items() if k != "may" and re.search(rf"\b{k}\b", t)}
    if re.search(r"\d\s*may\b|\bmay\s*\d", t):   # "may" is only a month next to a number
        nums.add(5)
    return nums

NEG_ZH = re.compile(r"不|毋|無須|无须|未|沒|没|勿|別|非")
NEG_EN = re.compile(r"\b(not|no|never|without|cannot|none|nobody)\b|n't")

def has_neg_en(text):
    text = re.sub(r"\bNo\.\s*\d", "", text)   # "No. 8" means "number 8", not a negation
    return bool(NEG_EN.search(text.lower()))

In [ ]:
# ---- Date roles: is this date an application deadline or an event date? ----
DEADLINE_ZH = re.compile(r"截止|報名|申請|遞交|交回|提交|日前|時前|或之前")
EVENT_ZH = re.compile(r"舉行|舉辦|進行|開始|開課|上課")
DEADLINE_EN = re.compile(r"\b(deadline|apply|application|applications|submit|return|enrol|enroll|enrolment|"
                         r"enrollment|register|registration|close|closes|closing|by|before)\b")
EVENT_EN = re.compile(r"\b(held|take place|takes place|start|starts|begin|begins|commence|commences)\b")

MONTH_RE = "|".join(MONTH_NAMES)

def dates_in_zh(clause):
    return {(cn_to_int(m), cn_to_int(d)) for m, d in re.findall(rf"([\d{CN_NUM}]+)月([\d{CN_NUM}]+)[日號号]", clause)}

def dates_in_en(clause):
    t = clause.lower()
    found = {(MONTHS[m], int(d)) for d, m in re.findall(rf"(?<!\d)(\d{{1,2}})(?:st|nd|rd|th)?\s+(?:of\s+)?({MONTH_RE})\b", t)}
    found |= {(MONTHS[m], int(d)) for m, d in re.findall(rf"\b({MONTH_RE})\s+(\d{{1,2}})(?!\d)", t)}
    return found

def role_of(clause, sentence, deadline_re, event_re):
    for text in (clause, sentence):          # look in the clause first, then the whole sentence
        if deadline_re.search(text):
            return "deadline"
        if event_re.search(text):
            return "event"
    return "other"

def date_roles(text, lang):
    roles = {"deadline": set(), "event": set(), "other": set()}
    if lang == "zh":
        sentences = [s for s in re.split(r"[。！？\n]", text) if s.strip()]
        for s in sentences:
            for clause in re.split(r"[，；、]", s):
                for d in dates_in_zh(clause):
                    roles[role_of(clause, s, DEADLINE_ZH, EVENT_ZH)].add(d)
    else:
        sentences = [s for s in re.split(r"(?<=[.!?])\s+(?=[A-Z])|\n", text) if s.strip()]
        for s in sentences:
            for clause in re.split(r"[;]|,\s+(?=and\b|but\b|while\b)", s):
                for d in dates_in_en(clause):
                    roles[role_of(clause.lower(), s.lower(), DEADLINE_EN, EVENT_EN)].add(d)
    return roles

def show_dates(ds):
    return ", ".join(f"{d} {MONTH_NAMES[m - 1].title()}" for m, d in sorted(ds))

def date_role_reasons(zh, en):
    rz, re_ = date_roles(zh, "zh"), date_roles(en, "en")
    names = {"deadline": "application deadline", "event": "event date"}
    out = []
    for role, label in names.items():
        if rz[role] and re_[role] and rz[role] != re_[role]:
            out.append(f"{label} differs: Chinese says {show_dates(rz[role])}, English says {show_dates(re_[role])}")
    return out

def rule_reasons(zh, en):
    reasons = []
    nz, ne = numbers_zh(zh), numbers_en(en)
    if nz != ne:
        missing, extra = sorted(nz - ne), sorted(ne - nz)
        parts = ([f"{missing} only in Chinese"] if missing else []) + ([f"{extra} only in English"] if extra else [])
        reasons.append("numbers differ: " + "; ".join(parts))
    if bool(NEG_ZH.search(zh)) != has_neg_en(en):
        reasons.append("one version says 'not/no/without' and the other doesn't")
    reasons += date_role_reasons(zh, en)
    return reasons

# quick check of the new rule
print(date_role_reasons("報名截止日期為11月20日，活動將於11月27日舉行。",
                        "The application deadline is 27 November, and the activity will be held on 20 November."))

### Improvements after the first evaluation
- A lone 一 or "one" used as a word (一名成人 = "an adult") is no longer compared as a number.
- Weekdays must match (星期一 = Monday).

In [ ]:
# ---- Improvement 1: "一" / "one" used as a word ("一名成人" = "an adult", "一經" = "once") is not a number to compare
_old_numbers_zh, _old_numbers_en = numbers_zh, numbers_en

def numbers_zh(text):
    text = re.sub(rf"(?<![{CN_NUM}])一(?![{CN_NUM}月日號时時])", "", text)   # drop a lone 一
    return _old_numbers_zh(text)

def numbers_en(text):
    return _old_numbers_en(re.sub(r"\bone\b", "", text, flags=re.I))

# ---- Improvement 2: weekdays must match (星期一 = Monday)
WEEKDAYS_EN = ["monday", "tuesday", "wednesday", "thursday", "friday", "saturday", "sunday"]
WEEKDAY_ZH = {"一": 0, "二": 1, "三": 2, "四": 3, "五": 4, "六": 5, "日": 6, "天": 6}

def weekday_reasons(zh, en):
    wz = {WEEKDAY_ZH[d] for d in re.findall(r"(?:星期|週|周|禮拜)([一二三四五六日天])", zh)}
    we = {i for i, d in enumerate(WEEKDAYS_EN) if re.search(rf"\b{d}s?\b", en.lower())}
    if wz and we and wz != we:
        show = lambda s: ", ".join(WEEKDAYS_EN[i].title() for i in sorted(s))
        return [f"weekday differs: Chinese says {show(wz)}, English says {show(we)}"]
    return []

_old_rule_reasons = rule_reasons
def rule_reasons(zh, en):
    return _old_rule_reasons(zh, en) + weekday_reasons(zh, en)

### Improvements from our own 60-case benchmark
- Text is normalised first (full-width digits, "noon" = 12 p.m., "midnight" = 12 a.m.).
- **Clock times** are compared as real times (下午2時 = 2 p.m.), so am/pm mix-ups are caught.
- Negation is compared more carefully: "unless" = 除非, "under 18" = 未滿18, and 不同 / 不論 / 不管 are not negations.
- "both" and "second" count as 2; 一併 / 一同 / 一起 / 一致 / 一般 are words, not numbers.
- "may" counts as a month only inside a date ("5 May", "May 5"), not in "may apply".

In [ ]:
import unicodedata

def normalise_text(text, lang):
    text = unicodedata.normalize("NFKC", text)
    if lang == "en":
        text = re.sub(r"\bnoon\b", "12 p.m.", text, flags=re.I)
        text = re.sub(r"\bmidnight\b", "12 a.m.", text, flags=re.I)
    return text

_prev_numbers_zh, _prev_numbers_en = numbers_zh, numbers_en

def numbers_zh(text):
    text = unicodedata.normalize("NFKC", text)
    text = re.sub(r"一併|一同|一起|一致|一般", "", text)
    return _prev_numbers_zh(text)

def numbers_en(text):
    t = normalise_text(text, "en").lower()
    # "may" followed by a verb is the modal, not the month ("the 12 may apply")
    t = re.sub(r"\bmay(?=\s+(?:reserve|apply|attend|submit|register|bring|enrol|make|book)\b)", "can", t)
    nums = _prev_numbers_en(t)
    nums |= {v for k, v in {"both": 2, "second": 2}.items() if re.search(rf"\b{k}\b", t)}
    return nums

def negation_pair(zh, en):
    # paired grammatical equivalents keep the meaning of the whole clause
    if re.search(r"\bunless\b", en, re.I):
        zh = zh.replace("除非", "")
    if re.search(r"\bunder\b|\bbelow\b|less than", en, re.I):
        zh = zh.replace("未滿", "")
    zh = zh.replace("不同", "").replace("不論", "").replace("不管", "")
    return bool(NEG_ZH.search(zh)), has_neg_en(en)

def clock_values(text, lang):
    # every clock time in the text, as minutes after midnight
    text = normalise_text(text, lang)
    values = []
    if lang == "en":
        for hour, minute, half in re.findall(r"\b(\d{1,2})(?:[:.](\d{2}))?\s*([ap])\.?m\.?", text, re.I):
            hour = int(hour)
            if 1 <= hour <= 12:
                values.append((hour % 12 + (12 if half.lower() == "p" else 0)) * 60 + int(minute or 0))
    else:
        # "下午2時至6時" -> "下午2時至下午6時"
        text = re.sub(r"(上午|下午|中午|正午|晚上|凌晨)(\d{1,2})[時点點](至|到)(\d{1,2})[時点點]",
                      lambda m: f"{m[1]}{m[2]}時{m[3]}{m[1]}{m[4]}時", text)
        for half, hour, minute in re.findall(r"(上午|下午|中午|正午|晚上|凌晨)(\d{1,2})[時点點](?:(\d{1,2})分?)?", text):
            hour = int(hour)
            if hour <= 12:
                values.append((hour % 12 + (12 if half in ("下午", "中午", "正午", "晚上") else 0)) * 60 + int(minute or 0))
    return set(values)

def rule_reasons(zh, en):
    reasons = []
    nz, ne = numbers_zh(zh), numbers_en(en)
    if nz != ne:
        missing, extra = sorted(nz - ne), sorted(ne - nz)
        parts = ([f"{missing} only in Chinese"] if missing else []) + ([f"{extra} only in English"] if extra else [])
        reasons.append("numbers differ: " + "; ".join(parts))
    zc, ec = clock_values(zh, "zh"), clock_values(en, "en")
    if zc and ec and zc != ec:
        fmt = lambda vs: ", ".join(f"{v // 60:02d}:{v % 60:02d}" for v in sorted(vs))
        reasons.append(f"clock times differ: Chinese {fmt(zc)}; English {fmt(ec)}")
    zn, en_neg = negation_pair(zh, en)
    if zn != en_neg:
        reasons.append("one version says 'not/no/without' and the other doesn't")
    return reasons + date_role_reasons(zh, en) + weekday_reasons(zh, en)

# quick checks
assert numbers_en("noon") == {12}
print(rule_reasons("活動於下午2時開始。", "The activity starts at 2 a.m."))

### Fixes after the first blind test
Our first run on real government notices (`fresh_test.csv`) caught 20 of 22 errors but raised 12 false alarms on 26 correct sentences.
Reading the false alarms showed bugs in our rules, which we fixed here:
- "$1,370" was read as 1 and 370. Thousands commas are now removed.
- Years written as 二○二六 were read digit by digit. They are now read as 2026.
- 分別, 特別, 不等 and similar words contain 別 / 不 but are not negations.
- List markers such as （三） are not numbers to compare.
- Clock times written in Chinese numerals (上午七時) are now compared too.

Because we used that file to find these bugs, it is now development data too. The final score comes from a second set of unseen notices.

In [ ]:
CN_DIGITS["○"] = 0
CN_NUM += "○"
_unit_cn_to_int = cn_to_int

def cn_to_int(s):
    # 二○二六 -> 2026 (digits written one by one, with no 十/百/千)
    if not s.isdigit() and len(s) > 1 and all(ch in CN_DIGITS for ch in s):
        return int("".join(str(CN_DIGITS[ch]) for ch in s))
    return _unit_cn_to_int(s)

_merged_numbers_zh, _merged_numbers_en, _merged_negation_pair = numbers_zh, numbers_en, negation_pair

def numbers_zh(text):
    text = re.sub(r"^\s*[（(][一二三四五六七八九十\d]+[)）]", "", text)   # list marker
    return _merged_numbers_zh(text)

def numbers_en(text):
    text = re.sub(r"(?<=\d),(?=\d{3}\b)", "", text)                     # 1,370 -> 1370
    return _merged_numbers_en(text)

NOT_NEGATION_ZH = re.compile(r"分別|特別|個別|類別|性別|級別|區別|不等|不少|非常|未來")

def negation_pair(zh, en):
    return _merged_negation_pair(NOT_NEGATION_ZH.sub("", zh), en)

_digit_clock_values = clock_values

def clock_values(text, lang):
    if lang == "zh":   # 上午七時十五分 -> 上午7時15分
        text = re.sub(rf"[{CN_NUM}]+(?=[時点點分])", lambda m: str(cn_to_int(m[0])), text)
    return _digit_clock_values(text, lang)

assert numbers_en("from $330 to $1,370") == {330, 1370}
assert clock_values("上午七時起", "zh") == {420}
assert numbers_zh("二○二六年") == {2026}

### Fixes after the final blind test
Our final blind test (`fresh_test_2.csv`) scored **19 of 21 errors caught with 7 false alarms on 26 correct sentences**. That is the number we report.
Afterwards we fixed the false alarms that had a clear cause. Results printed below for that file are therefore *after-fix*, not blind.
- Number words: 兩年一度 = "biennially", 半費 = "50 per cent", 第三季 = "third quarter", 五人足球 = "futsal".
- 組別, 識別 and 辨別 contain 別 but are not negations.
- Web links are removed before the text goes to the pretrained models (a link ending /tc/ against /en/ looked like a contradiction).

In V8 these fixes were rewritten as **bounded normalisation**: instead of deleting words like 兩年一度
or a web link, the checker turns them into facts (a frequency, a link with its language part removed)
and compares those facts too. So "held biennially" vs 一年一度 is still caught, and so is a link that
points to a different page.

In [ ]:
# How rule warnings and model scores are combined (the same idea since v2)
def model_reasons(sim, contra, sim_t, nli_t):
    reasons = []
    if sim < sim_t:
        reasons.append(f"meaning may differ (similarity {sim:.2f})")
    if contra > nli_t:
        reasons.append(f"the two versions may contradict each other ({contra:.0%})")
    return reasons

def checker_flag(zh, en, sim=None, contra=None):
    model_zh, model_en = normalise_text(zh, "zh"), normalise_text(en, "en")
    sim = similarity(model_zh, model_en) if sim is None else sim
    contra = contradiction(model_zh, model_en) if contra is None else contra
    reasons = rule_reasons(zh, en) + model_reasons(sim, contra, SIM_T, NLI_T)
    return bool(reasons), reasons

"""Bounded lexical normalisation adapted from teammate V5.

Frequency and destination information are retained for explicit comparison.
Install after the V4 rule and model definitions in a fresh namespace.
"""
import re
import unicodedata
from urllib.parse import urlsplit, parse_qsl, urlencode

EN_ORDINALS = {'third':3, 'fourth':4, 'fifth':5, 'sixth':6, 'seventh':7,
               'eighth':8, 'ninth':9, 'tenth':10}
URL_RE = re.compile(r'(?<![a-z0-9_@.])(?:https?://)?(?:[a-z0-9-]+\.)+[a-z]{2,}(?::\d+)?(?:/[^\s<>"「」『』，。；！？）)]*)?', re.I)
LOCALES = {'en','tc','sc','zh','zh-hk','zh-tw','zh-cn','en-hk','en-us','en-gb'}

def notice_urls(text):
    return [m.group(0).rstrip('.,;:!?') for m in URL_RE.finditer(unicodedata.normalize('NFKC', text))]

def canonical_url(value):
    u = urlsplit(value if '://' in value else 'https://'+value)
    parts = u.path.split('/')
    if len(parts)>1 and parts[1].lower() in LOCALES:
        parts.pop(1)
    query = [(k,v) for k,v in parse_qsl(u.query,keep_blank_values=True)
             if not (k.lower() in {'lang','locale','language'} and v.lower() in LOCALES)]
    # Path, content identifiers and non-language query values remain significant.
    return (u.netloc.lower(), '/'.join(parts).rstrip('/'), urlencode(sorted(query)), u.fragment)

def without_urls(text):
    return URL_RE.sub('', text)

def link_evidence(zh,en):
    z,e=notice_urls(zh),notice_urls(en)
    if z and e and {canonical_url(x) for x in z}!={canonical_url(x) for x in e}:
        return [{'category':'link destination', 'chinese_value':str([canonical_url(x) for x in z]),
                 'english_value':str([canonical_url(x) for x in e]),
                 'chinese_span':' | '.join(z),'english_span':' | '.join(e)}]
    return []

def frequency_facts(text,lang):
    t=unicodedata.normalize('NFKC',text).lower()
    found=[]
    if lang=='zh':
        for m in re.finditer(r'([一兩二三四\d]+)年一度|每([一兩二三四\d]+)年',t):
            found.append((cn_to_int(m.group(1) or m.group(2)),m.group(0)))
        for m in re.finditer(r'每年|一年一度',t):found.append((1,m.group(0)))
    else:
        names={'annual':1,'annually':1,'yearly':1,'biennial':2,'biennially':2,
               'triennial':3,'triennially':3,'quadrennial':4,'quadrennially':4}
        for m in re.finditer(r'\b(?:'+ '|'.join(names) +r')\b',t):found.append((names[m.group()],m.group()))
        for m in re.finditer(r'\bevery (one|two|three|four|\d+) years?\b',t):
            value={'one':1,'two':2,'three':3,'four':4}.get(m.group(1))
            found.append((int(m.group(1)) if value is None else value,m.group()))
        for m in re.finditer(r'\bevery other year\b',t):found.append((2,m.group()))
    values={v for v,_ in found}
    return {'value':next(iter(values)),'span':found[0][1]} if len(values)==1 else None

def frequency_evidence(zh,en):
    z,e=frequency_facts(zh,'zh'),frequency_facts(en,'en')
    if z and e and z['value']!=e['value']:
        return [{'category':'event frequency (years between occurrences)',
                 'chinese_value':z['value'],'english_value':e['value'],
                 'chinese_span':z['span'],'english_span':e['span']}]
    return []

def lexical_text(text,lang):
    t=unicodedata.normalize('NFKC',text)
    if lang=='zh':
        t=re.sub(r'([一兩二三四\d]+)年一度|每([一兩二三四\d]+)年',
                 lambda m:f"每{cn_to_int(m.group(1) or m.group(2))}年",t)
        t=re.sub(r'每年', '每1年',t)
        t=re.sub(r'半費|半價','50%',t)
    else:
        freq={'annual':1,'annually':1,'yearly':1,'biennial':2,'biennially':2,
              'triennial':3,'triennially':3,'quadrennial':4,'quadrennially':4}
        t=re.sub(r'\b(?:'+ '|'.join(freq) +r')\b',lambda m:f"every {freq[m.group().lower()]} years",t,flags=re.I)
        t=re.sub(r'\bevery other year\b','every 2 years',t,flags=re.I)
        t=re.sub(r'\bevery (one|two|three|four) years?\b',
                 lambda m:f"every {dict(one=1,two=2,three=3,four=4)[m.group(1).lower()]} years",t,flags=re.I)
        t=re.sub(r'\bfutsal\b','5-a-side football',t,flags=re.I)
        t=re.sub(r'\bhalf[- ]price\b|\bhalf (?:the )?(?:normal |usual |standard )?(?:price|fee|fare)\b','50 percent',t,flags=re.I)
    return t

_merge_base_numbers_zh = getattr(numbers_zh, '_merge_original', numbers_zh)
_merge_base_numbers_en = getattr(numbers_en, '_merge_original', numbers_en)
_merge_base_negation_pair = getattr(negation_pair, '_merge_original', negation_pair)
_merge_base_similarity = getattr(similarity, '_merge_original', similarity)
_merge_base_contradiction = getattr(contradiction, '_merge_original', contradiction)

def numbers_zh(text):
    return _merge_base_numbers_zh(lexical_text(without_urls(text),'zh'))

def numbers_en(text):
    t=lexical_text(without_urls(text),'en')
    nums=_merge_base_numbers_en(t)
    nums|={v for k,v in EN_ORDINALS.items() if re.search(rf'\b{k}\b',t,re.I)}
    if re.search(r'\bfirst (?:quarter|floor|day|month|year)\b',t,re.I):nums.add(1)
    return nums

def negation_pair(zh,en):
    return _merge_base_negation_pair(re.sub(r'組別|識別|辨別','',zh),en)

def model_text(text,lang):
    return lexical_text(without_urls(normalise_text(text,lang)),lang).strip()

def similarity(zh,en):
    z,e=model_text(zh,'zh'),model_text(en,'en')
    return 1.0 if not z and not e else _merge_base_similarity(z,e)

def contradiction(zh,en):
    z,e=model_text(zh,'zh'),model_text(en,'en')
    return 0.0 if not z and not e else _merge_base_contradiction(z,e)

def ambiguity_evidence(zh,en):
    return frequency_evidence(zh,en)+link_evidence(zh,en)

# Re-running this cell must not wrap a wrapper and recurse indefinitely.
for _wrapped, _original in [(numbers_zh, _merge_base_numbers_zh),
                            (numbers_en, _merge_base_numbers_en),
                            (negation_pair, _merge_base_negation_pair),
                            (similarity, _merge_base_similarity),
                            (contradiction, _merge_base_contradiction)]:
    _wrapped._merge_original = _original

### V8: evidence rules
New checks that compare **what each version tells the reader to do**:
- **Actions:** bring, submit, upload, reserve, reply, return, attend, pay, register. If the Chinese says
  攜帶 (must bring) and the English says "may bring", the duty has changed.
- **Relations:** who must do what for whom (a proxy bringing *your* original ID vs a copy), and which
  condition a rule depends on.
- Every rule **abstains** when it cannot find the feature on both sides, instead of guessing.

Thresholds are fixed here and are not tuned on any test labels: similarity below **0.40**, or
contradiction above **0.65**, gives a firm warning.

In [ ]:
"""Conservative evidence rules for HK bilingual notices.
No case IDs, benchmark labels or reference rationales are used.
Each rule abstains when it cannot resolve a supported feature on both sides.
This finite grammar is not a general semantic parser.
"""
import re
import unicodedata

ACTIONS = {
    'bring': (r'\bbring\b', r'攜同|攜帶|帶備|帶同'),
    'submit': (r'\bsubmit\b', r'提交|遞交|交回'),
    'upload': (r'\bupload\b', r'上載|上傳'),
    'reserve': (r'\b(?:reserve|book)\b', r'預留|預訂'),
    'reply': (r'\b(?:reply|respond)\b', r'回覆|回復'),
    'return': (r'\breturn\b', r'歸還'),
    'attend': (r'\battend\b', r'出席'),
    'pay': (r'\bpay\b', r'繳付|支付'),
    'register': (r'\bregister\b', r'報名|登記'),
}

def _norm(s):
    return unicodedata.normalize('NFKC', s).replace('’', "'")

def _hit(pattern, text):
    return re.search(pattern, text, re.I)

def _fact(value, match):
    return {'value': value, 'span': match.group(0)} if match else None

def _unique(patterns, text):
    found = [(value, m) for value, pattern in patterns for m in re.finditer(pattern, text, re.I)]
    if len({v for v, _ in found}) != 1:
        return None
    value, match = found[0]
    return _fact(value, match)

def _main_clause(text, lang):
    # Match the consequence, not a permission mentioned inside an exception.
    if lang == 'en':
        text = re.split(r'\bunless\b|\bonly if\b', text, maxsplit=1, flags=re.I)[0]
        if _hit(r'^\s*(?:if|when)\b', text) and ',' in text:
            text = text.split(',', 1)[1]
    else:
        if '否則' in text:
            text = text.split('否則', 1)[1]
        elif _hit(r'^\s*(?:如|若|假如)', text) and ',' in text:
            text = text.split(',', 1)[1]
    return text

def _modal_for_action(text, action_pattern, lang):
    actions = list(re.finditer(action_pattern, text, re.I))
    if len(actions) != 1:
        return None
    action = actions[0]
    prefix = text[:action.start()]
    prefix = re.split(r'[,;。]', prefix)[-1]
    patterns = ([("not required", r'\b(?:need not|do(?:es)? not (?:need|have) to|(?:is|are) not required to)\s*$'),
                 ("prohibited", r'\b(?:must not|cannot|can\x27t)\s*$'),
                 ("required", r'\b(?:must|need(?:s)? to|have to|has to|(?:is|are) required to)\s*$'),
                 ("permitted", r'\b(?:may|can|(?:is|are) allowed to)\s*$')]
                if lang == 'en' else
                [("not required", r'(?:毋須|無須|不必|不須)[^,。;]{0,12}$'),
                 ("prohibited", r'(?:不得|不可|禁止)[^,。;]{0,12}$'),
                 ("required", r'(?<![毋無不])(?:必須|須|必需)[^,。;]{0,12}$'),
                 ("permitted", r'(?<![不])(?:可以|可)[^,。;]{0,12}$')])
    if lang == 'zh':
        # In 須待...才可報名 the obligation governs waiting; the closest modal
        # 可 governs registering. Do not assign the earlier 須 to the action.
        candidates=[(m.start(),value,m) for value,pattern in patterns
                    for m in re.finditer(pattern,prefix,re.I)]
        if candidates:
            _,value,m=max(candidates,key=lambda item:item[0])
            return {'value':value,'span':m.group(0)+action.group(0)}
    for value, pattern in patterns if lang == 'en' else []:
        m = _hit(pattern, prefix)
        if m:
            return {'value': value, 'span': m.group(0) + action.group(0)}
    if lang == 'en' and _hit(r'^\s*(?:please\s+)?$', prefix):
        return {'value': 'required', 'span': text[:action.end()].strip()}
    if lang == 'zh' and _hit(r'^\s*請(?:於|在)?[^,。;]*$', prefix):
        return {'value': 'required', 'span': prefix.strip()+action.group(0)}
    return None


def semantic_evidence(chinese, english):
    """Return paired evidence for supported, non-equivalent semantic features."""
    zh, en = _norm(chinese), _norm(english)
    evidence = []
    def compare(category, z, e):
        if z and e and z['value'] != e['value']:
            evidence.append({'category': category, 'chinese_value': z['value'],
                             'english_value': e['value'], 'chinese_span': z['span'],
                             'english_span': e['span']})

    zm, em = _main_clause(zh, 'zh'), _main_clause(en, 'en')
    for action, (ep, zp) in ACTIONS.items():
        compare('obligation for '+action, _modal_for_action(zm,zp,'zh'), _modal_for_action(em,ep,'en'))

    # The approved exception is removed from both sides before comparing the
    # ordinary submission routes. An added paper option outside the exception
    # is a substantive change even when the document mentions paper elsewhere.
    if _hit(r'\bunless\b',en) and '否則' in zh and _hit(r'\bsubmit\b',em) and _hit(r'提交|遞交',zm):
        def channels(text, lang):
            patterns = [('online',r'\bonline\b'),('paper',r'\bpaper\b')] if lang=='en' else [('online',r'網上|線上'),('paper',r'紙本')]
            found = [(name,m.group(0)) for name,p in patterns for m in re.finditer(p,text,re.I)]
            return {'value': ', '.join(sorted({n for n,_ in found})), 'span': text.strip()} if found else None
        compare('submission options outside the approved exception',channels(zm,'zh'),channels(em,'en'))

    # Restricted eligibility: compare the role explicitly governed by "only".
    groups = {'members':(r'members?',r'會員'), 'volunteers':(r'volunteers?',r'義工|志願者'),
              'performers':(r'performers?',r'演出者'), 'residents':(r'residents?',r'居民')}
    eg = zg = None
    for value,(ep,zp) in groups.items():
        m = _hit(r'\bonly\s+('+ep+r')\b',en)
        if m: eg = _fact(value,m)
        m = _hit(r'(?:只有|只限)[^,。;]{0,35}(?:'+zp+r')(?=可|須|才|均|必)',zh)
        if m: zg = _fact(value,m)
    m = _hit(r'(?:所有|全體)參加者',zh)
    if m: zg = _fact('all participants',m)
    compare('eligible group',zg,eg)

    compare('day-count basis',
            _unique([('working days',r'工作[天日]'),('calendar days',r'曆日|日曆日')],zh),
            _unique([('working days',r'\b(?:working|business) days?\b'),('calendar days',r'\bcalendar days?\b')],en))
    compare('deadline starts at',
            _unique([('receipt',r'(?:收到|接獲)[^,。;]{0,20}後'),('issuance',r'(?:發出|寄出|簽發)後')],zh),
            _unique([('receipt',r'\bafter\s+(?:receiving|receipt of)\b'),('issuance',r'\bafter\s+(?:issuing|issuance of)|\b(?:issued|sent)\b[^,;]{0,12}\bafter\b')],en))

    # Explicit ordered boundary. Do not infer comparators from standalone 未/不.
    zn = r'[零〇一二兩三四五六七八九十百\d]+'
    compare('inclusive/exclusive threshold',
            _unique([('>=',zn+r'(?:號|歲)?或以上'),('>',r'高於'+zn+r'(?:號|歲)?')],zh),
            _unique([('>=',r'\b\d+(?:\s+years?)?\s+or\s+(?:higher|above|over)\b'),('>',r'\b(?:higher than|above)\s+(?:No\.\s*)?\d+\b')],en))
    # This compares a temporal direction only when the weekday anchor agrees.
    days = ['monday','tuesday','wednesday','thursday','friday','saturday','sunday']
    zdays = '一二三四五六日'
    for idx, day in enumerate(days):
        zp = r'(?:星期|週|周|禮拜)'+zdays[idx]
        if _hit(zp,zh) and _hit(r'\b'+day+r'\b',en):
            compare('weekday deadline boundary',
                    _unique([('by',zp+r'或之前'),('after',zp+r'(?:之後|後)'),('before',zp+r'之前')],zh),
                    _unique([('by',r'\bby\s+'+day+r'\b'),('after',r'\bafter\s+'+day+r'\b'),('before',r'\bbefore\s+'+day+r'\b')],en))

    # Paired enumerations of the same supported documents: and/or changes requirements.
    documents=[(r'proof of address',r'住址證明'),(r'(?:a\s+)?student ID',r'學生證')]
    if all(_hit(ep,en) and _hit(zp,zh) for ep,zp in documents):
        compare('document combination',
                _unique([('both',r'住址證明(?:及|和|與)學生證'),('either',r'住址證明或學生證')],zh),
                _unique([('both',r'proof of address\s+and\s+(?:a\s+)?student ID'),('either',r'proof of address\s+or\s+(?:a\s+)?student ID')],en))

    compare('application limit applies per',
            _unique([('household',r'每(?:個|一個)?住戶'),('address',r'每(?:個|一個)?地址')],zh),
            _unique([('household',r'\bper household\b'),('address',r'\bper address\b')],en))
    # Compare the action's address condition. A shared-address subject plus
    # a different-address-only condition is narrower than shared-address permission.
    if _hit(r'分開(?:提交|遞交)',zh) and _hit(r'\b(?:submit|file) separate',en):
        zp = _hit(r'(?:只有|只限)[^,。;]{0,12}(?:住址|地址)不同',zh)
        ep = _hit(r'\bsharing an address\b|\b(?:at|with) the same address\b',en)
        if zp and ep and not _hit(r'\bdifferent addresses\b',en):
            compare('address condition for separate applications',_fact('different addresses required',zp),_fact('shared address permitted',ep))

    # Explicit session role, as distinct from a role mentioned in an antecedent.
    compare('role of required session',
            _unique([('new role',r'新崗位的(?:簡介會|培訓)'),('former role',r'(?:原有|舊)崗位的(?:簡介會|培訓)')],zh),
            _unique([('new role',r'\b(?:session|induction|training) for (?:their |the |your )?new role\b'),
                     ('former role',r'\b(?:session|induction|training) for (?:their |the |your )?(?:former|old|previous) role\b')],en))
    # Both versions must explicitly describe queue membership at a cutoff.
    if _hit(r'排隊|隊伍',zh) and _hit(r'\bqueue\b',en):
        compare('queue membership at cutoff',
                _unique([('already queued',r'\d+時已在排隊'),('join after cutoff',r'\d+時後才加入隊伍')],zh),
                _unique([('already queued',r'\balready in the queue at\b'),('join after cutoff',r'\b(?:join|joining) the queue after\b')],en))
    return evidence


def semantic_reasons(chinese, english):
    return [f"{x['category']} differs: Chinese {x['chinese_value']} [{x['chinese_span']}]; English {x['english_value']} [{x['english_span']}]"
            for x in semantic_evidence(chinese,english)]


"""Supported relation checks. Unrecognised constructions remain unresolved.

Developed on the reused 60-case suite and explicit contrast fixtures; not a
general parser and not evidence of general 100% translation accuracy.
"""
import re
import unicodedata

def relation_evidence(chinese,english):
    z=unicodedata.normalize('NFKC',chinese)
    e=unicodedata.normalize('NFKC',english).replace('’',"'")
    out=[]
    def single(patterns,t):
        hits=[(v,m.group()) for v,p in patterns for m in re.finditer(p,t,re.I)]
        return {'value':hits[0][0],'span':hits[0][1]} if len({v for v,_ in hits})==1 else None
    def compare(category,zf,ef):
        if zf and ef and zf['value']!=ef['value']:
            out.append({'category':category,'chinese_value':zf['value'],'english_value':ef['value'],
                        'chinese_span':zf['span'],'english_span':ef['span']})

    if re.search(r'住址證明',z) and re.search(r'\bproof\b',e,re.I):
        compare('parent relationship to proof',
                single([('named person',r'以家長名義(?:發出|簽發)|家長名下'),('issuer',r'由家長(?:簽發|發出)')],z),
                single([('named person',r"in (?:your |the )?parent(?:'s|s') name|issued (?:in|under) (?:your |the )?parent(?:'s|s') name"),
                        ('issuer',r'issued by (?:your |the )?parents?')],e))

    # Explicit proxy subject anchors "your ID" to the represented recipient,
    # and "their own ID" to the proxy. No free pronoun guessing elsewhere.
    if '代領人' in z and re.search(r'\b(?:the|your) proxy\b',e,re.I):
        for role,zowner,eowner in [('recipient',r'(?:收件人|領取人)的',r'(?:your|the recipient\x27s)'),
                                  ('proxy',r'代領人(?:本人)?的',r'(?:their own|the proxy\x27s own)')]:
            zp=[(form,zowner+r'(?:身分|身份)證明文件'+word) for form,word in [('original','正本'),('copy','副本')]]
            ep=[(form,r'\b'+word+r' of '+eowner+r' (?:ID|identity document)\b|\b'+eowner+r' '+word+r' (?:ID|identity document)\b')
                for form,word in [('original','original'),('copy','copy')]]
            compare(role+' ID document form',single(zp,z),single(ep,e))

    # Compare only explicitly supported complete collection conditions.
    if re.search(r'委託他人代領|代領',z) and re.search(r'\bproxy\b',e,re.I):
        compare('condition for proxy collection',
                single([('cannot collect',r'(?:如|若)未能親身取件'),
                        ('cannot attend original slot',r'(?:如|若)未能在原定時段親身取件')],z),
                single([('cannot collect',r"\bif you (?:cannot|can't|are unable to) collect (?:your |the )?(?:parcel|item|package)[.!?]?\s*$"),
                        ('cannot attend original slot',r"\bif you (?:cannot|can't|are unable to) collect (?:your |the )?(?:parcel|item|package) (?:at|during|in) (?:the |your )?(?:original|originally scheduled) (?:slot|time|collection slot|appointment)[.!?]?\s*$")],e))

    if '同行者' in z and '照顧者' in z and re.search(r'\bcompanions?\b',e,re.I) and re.search(r'\bcarers?\b',e,re.I):
        compare('carer requirement for companions',
                single([('both must be carers',r'兩名同行者均為其照顧者'),
                        ('one carer sufficient',r'其中一名同行者為其照顧者')],z),
                single([('both must be carers',r'\bboth companions are (?:their |your )?carers\b'),
                        ('one carer sufficient',r'\b(?:at least )?one companion is (?:their |your )?carer\b')],e))

    compare('registration timing for other eligible applicants',
            single([('registration allowed',r'其他合資格人士仍可報名'),
                    ('after priority period only',r'其他合資格人士須待優先階段結束後才可報名')],z),
            single([('registration allowed',r'\bOther eligible applicants can still register[.!?]?\s*$'),
                    ('after priority period only',r'\bOther eligible applicants (?:can|may) (?:only )?register (?:only )?after the priority (?:period|phase) ends[.!?]?\s*$')],e))
    return out


from scipy.optimize import linear_sum_assignment

def split_zh(text):
    # keep conditions attached; split separate availability statements after a comma
    return [s.strip() for s in re.split(r"(?<=[。！？；])|\n|(?<=，)(?=(?:不設|設有))", text) if s and s.strip()]

def split_en(text):
    # split after . ! ? only when the next sentence starts with a capital letter,
    # so "5 p.m. on" and "No. 8" stay together
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+(?=[A-Z])|(?<=;)\s*|\n", text) if s and s.strip()]

def alignment_text(text, lang):
    # hide digits while matching, so a sentence is not matched just because its number is the same
    return re.sub(r"\d+", " ", model_text(text, lang))

def check_notice(chinese_text, english_text):
    zh, en = split_zh(chinese_text), split_en(english_text)
    if not zh or not en:
        return pd.DataFrame([{"status": "⚠️ Check", "chinese": chinese_text, "english": english_text, "why": "Paste both versions first."}])
    ez = labse.encode([alignment_text(s, "zh") for s in zh], convert_to_tensor=True, normalize_embeddings=True)
    ee = labse.encode([alignment_text(s, "en") for s in en], convert_to_tensor=True, normalize_embeddings=True)
    sims = util.cos_sim(ez, ee).cpu().numpy()
    # one-to-one matching over the whole notice, so two Chinese sentences cannot reuse one English sentence
    ii, jj = linear_sum_assignment(-sims)
    matches = dict(zip(ii.tolist(), jj.tolist()))
    used, out = set(), []
    for i, z in enumerate(zh):
        j = matches.get(i)
        if j is None or sims[i, j] < 0.25:
            out.append({"status": "⚠️ Check", "chinese": z, "english": "", "why": "Needs review: no confident English counterpart"})
            continue
        used.add(j)
        flagged, reasons = checker_flag(z, en[j])
        out.append({"status": "⚠️ Check" if flagged else "✅ OK", "chinese": z, "english": en[j], "why": "; ".join(reasons)})
    for j, e in enumerate(en):
        if j not in used:
            out.append({"status": "⚠️ Check", "chinese": "", "english": e, "why": "Needs review: no confident Chinese counterpart"})
    for reason in date_role_reasons(chinese_text, english_text):   # whole-notice date roles
        if not any(reason in row["why"] for row in out):
            out.append({"status": "⚠️ Check", "chinese": "(whole notice)", "english": "(whole notice)", "why": reason})
    return pd.DataFrame(out)
# Fixed production thresholds; no tuning on reference labels.
SIM_T, NLI_T = 0.40, 0.65
CHECKER_VERSION = "merged-v5-candidate"
RULE_SOURCE_HASHES = {'semantic_checks.py': 'bea107d8d4daec36ec8f2dc6895d517cb6703b1f302d1e4400c61d54cef7fcea', 'relation_checks.py': '8346aa005231e6c01e9c468f3a25cc9463f4573dd8be26babb5c03cd72d3bce9', 'ambiguity_normalisation.py': 'dda745a63cf1dc2563d100c95a6ccd766458bd45fcd911f19c03ea41ede094ba'}
SEMANTIC_RULES_SHA256 = RULE_SOURCE_HASHES["semantic_checks.py"]

def paired_evidence(zh,en):
    return semantic_evidence(zh,en)+ambiguity_evidence(zh,en)+relation_evidence(zh,en)

def evidence_reasons(zh,en):
    return [f"{x['category']} differs: Chinese {x['chinese_value']} [{x['chinese_span']}]; English {x['english_value']} [{x['english_span']}]"
            for x in paired_evidence(zh,en)]

def checker_flag(zh,en,sim=None,contra=None):
    sim = similarity(zh,en) if sim is None else sim
    contra = contradiction(zh,en) if contra is None else contra
    reasons = rule_reasons(zh,en)+model_reasons(sim,contra,SIM_T,NLI_T)+evidence_reasons(zh,en)
    return bool(reasons),list(dict.fromkeys(reasons))

_merged_base_check_notice = check_notice

def check_notice(chinese_text,english_text):
    result = _merged_base_check_notice(chinese_text,english_text)
    result["semantic_evidence"] = [paired_evidence(row.chinese,row.english)
        if row.chinese and row.english and row.chinese != "(whole notice)" else []
        for row in result.itertuples()]
    return result

print("Merged candidate loaded; thresholds:", SIM_T, NLI_T)
print("Rule source hashes:", RULE_SOURCE_HASHES)

### V8: whole-notice patterns
Some short notices (a badge pick-up order, which gate to use, which volunteers do which duty) are read as a
whole: each language is parsed **independently** into a small record (kind of notice, topic, facts), and the
two records are compared. If a notice does not fit a known pattern completely, it goes to the checks above
unchanged.

In [ ]:
"""V8 bounded, whole-notice semantic grammar.

Parses each language independently. No benchmark IDs, labels, examples or
rationales are consulted. Every pattern must cover the entire notice; unknown
clauses or constructions fall back to the frozen checker. This is a finite
notice grammar, not general translation understanding or weight fine-tuning.
"""
import re
import unicodedata
from dataclasses import dataclass

@dataclass
class Notice:
    kind: str
    topic: str
    facts: dict
    text: str

# Topics are noun phrases, not arbitrary sentences. Extra clauses must not be
# swallowed as a topic and thereby evade the inherited checker.
ET = r'(?P<topic>[^.,;:!?\n]+?)'
ZT = r'(?P<topic>[^,。;:!?\n]+?)'
MONTH_NAMES = 'January February March April May June July August September October November December'.split()
MONTH_MAP = {s.lower(): i+1 for i,s in enumerate(MONTH_NAMES)}
MONTH_PATTERN = '|'.join(MONTH_NAMES)

def norm(text):
    return unicodedata.normalize('NFKC',text).replace('，',',').replace('：',':').strip()

def clock(h, half, minute=None):
    h=int(h); minute=int(minute or 0)
    if not 1 <= h <= 12 or not 0 <= minute < 60: raise ValueError('invalid clock')
    return f'{h%12+(12 if half.lower() in ("p","下午","晚上","中午","正午") else 0):02d}:{minute:02d}'

def ct(prefix, lang):
    if lang=='en':
        return rf'(?P<{prefix}h>\d{{1,2}})(?::(?P<{prefix}min>\d{{2}}))?\s*(?P<{prefix}half>[ap])\.?m\.?'
    return rf'(?P<{prefix}half>上午|下午|晚上|凌晨|中午|正午)(?P<{prefix}h>\d{{1,2}})[時點](?:(?P<{prefix}min>\d{{1,2}})分)?'

def timeval(d,p=''):
    return clock(d[p+'h'],d[p+'half'],d.get(p+'min'))

def datepat(p,lang):
    if lang=='en': return rf'(?P<{p}d>\d{{1,2}})\s+(?P<{p}m>{MONTH_PATTERN})'
    return rf'(?P<{p}m>\d{{1,2}})月(?P<{p}d>\d{{1,2}})日'

def dateval(d,p,lang):
    m=MONTH_MAP[d[p+'m'].lower()] if lang=='en' else int(d[p+'m'])
    day=int(d[p+'d'])
    if not 1<=m<=12 or not 1<=day<=31: raise ValueError('invalid date')
    return f'{m:02d}-{day:02d}'

RULES = {'en':[], 'zh':[]}
def add(lang,kind,pattern,convert):
    RULES[lang].append((kind,re.compile(pattern+r'[。.]?',re.I),convert))

# Order is represented as two named actions, never the polarity of one token.
for lang in RULES:
    if lang=='en':
        for action,other,kind in [('report to the hall','collecting your badge','badge order'),('submit your request','creating an account','account order')]:
            first,second=('collect badge','report to hall') if kind=='badge order' else ('create account','submit request')
            add(lang,kind,rf'For the {ET},\s*{action} (?P<order>after|before) {other}',
                lambda d,a=first,b=second:{'action order':f'{a} before {b}' if d['order'].lower()=='after' else f'{b} before {a}'})
    else:
        for kind,a,b in [('badge order','領取名牌','到禮堂報到'),('account order','開立帳戶','提交申請')]:
            names={a:'collect badge' if kind=='badge order' else 'create account',b:'report to hall' if kind=='badge order' else 'submit request'}
            add(lang,kind,rf'(?:出席|參與|辦理){ZT}(?:服務)?時,\s*請先(?P<a>{a}|{b}),\s*再(?P<b>{a}|{b})',
                lambda d,n=names:{'action order':n[d['a']]+' before '+n[d['b']]})
        add(lang,'badge order',rf'請領取{ZT}名牌後才到禮堂報到',lambda d:{'action order':'collect badge before report to hall'})
        add(lang,'account order',rf'請開立帳戶後才提交{ZT}(?:服務)?申請',lambda d:{'action order':'create account before submit request'})

# Receipt validity: month offset and boundary, with 最後一日 treated as a phrase.
add('en','receipt expiry',rf'A receipt for the {ET} is valid until the end of the month (?P<anchor>in which|after) it was issued',lambda d:{'receipt expiry':'end of issue month' if d['anchor'].lower()=='in which' else 'end of following month'})
add('zh','receipt expiry',rf'{ZT}的收據有效至發出(?P<anchor>當月月底|後下一個月的月底)',lambda d:{'receipt expiry':'end of issue month' if d['anchor']=='當月月底' else 'end of following month'})
add('zh','receipt expiry',rf'{ZT}收據的有效期於發出(?P<anchor>當月|後下一個月)最後一日屆滿',lambda d:{'receipt expiry':'end of issue month' if d['anchor']=='當月' else 'end of following month'})

# Necessary-condition wording, not an assertion of sufficient permission.
exception_value=lambda d:{'side-door exception':'staff presence is necessary for opening' if d['cond'].lower() in ('unless','除非','只在') else 'closed even with staff present'}
for p in [rf'During the {ET},\s*the side door is closed (?P<cond>unless|even when) a staff member is on duty there',rf'The side door for the {ET} is closed (?P<cond>unless|even when) a staff member is on duty there']:
    add('en','door exception',p,exception_value)
for p in [rf'{ZT}期間,\s*(?P<cond>除非|即使)有職員在側門當值,(?:否則|仍)側門關閉',rf'{ZT}的側門(?P<cond>除非|即使)有職員在該處當值,(?:否則|仍)關閉',rf'{ZT}(?:期間,\s*|的)側門(?P<cond>只在)有職員於該處當值時開放']:
    add('zh','door exception',p,exception_value)

# Group obligation stays attached to the group; walk-in permission is separate.
add('en','group booking',rf'The {ET} accepts walk-ins\.\s*Group visits (?P<modal>must be booked in advance|do not need advance booking)',lambda d:{'individual walk-ins':'allowed','group advance booking':'required' if d['modal'].lower().startswith('must') else 'not required'})
for p in [rf'{ZT}接受即場到訪,但團體到訪(?P<modal>須|毋須)預約',rf'個別人士可即場到訪{ZT}。團體則(?P<modal>須|毋須)事先預約']:
    add('zh','group booking',p,lambda d:{'individual walk-ins':'allowed','group advance booking':'required' if d['modal']=='須' else 'not required'})

# Role assignment maps action and object to actor, regardless of clause order.
ACTORS_EN='class teachers|prefects|reception staff|volunteers'
ACTORS_ZH='班主任|領袖生|接待職員|義工'
ACTOR_MAP={'班主任':'class teachers','領袖生':'prefects','接待職員':'reception staff','義工':'volunteers'}
add('en','duties',rf'(?:For|At) the {ET},\s*(?P<check>{ACTORS_EN}) check the list and (?P<distribute>{ACTORS_EN}) distribute the materials',lambda d:{'list checking actor':d['check'].lower(),'materials distribution actor':d['distribute'].lower()})
for p in [rf'{ZT}(?:期間,|的)(?P<check>{ACTORS_ZH})負責核對名單,(?P<distribute>{ACTORS_ZH})負責派發物資',rf'{ZT}期間的分工如下:名單由(?P<check>{ACTORS_ZH})核對。物資由(?P<distribute>{ACTORS_ZH})派發',rf'在{ZT},名單由(?P<check>{ACTORS_ZH})核對,物資由(?P<distribute>{ACTORS_ZH})派發']:
    add('zh','duties',p,lambda d:{'list checking actor':ACTOR_MAP[d['check']],'materials distribution actor':ACTOR_MAP[d['distribute']]})

add('en','contact update',rf'For the {ET},\s*you may update your contact number (?P<cond>without changing|only after changing) the name on your account',lambda d:{'name change prerequisite':'not required' if d['cond'].lower().startswith('without') else 'required before phone update'})
add('zh','contact update',rf'辦理{ZT}時,可更新聯絡電話,(?P<cond>毋須更改|但須先更改)帳戶姓名',lambda d:{'name change prerequisite':'not required' if d['cond'].startswith('毋須') else 'required before phone update'})
add('zh','contact update',rf'帳戶姓名可保留不變,仍可在辦理{ZT}時更新聯絡電話',lambda d:{'name change prerequisite':'not required'})

# Corrections bind current and superseded values instead of comparing value sets.
add('en','room correction',rf'{ET} update:\s*disregard Room (?P<old>\d+) (?:in the earlier circular|on the earlier sign)\.\s*Use Room (?P<current>\d+)',lambda d:{'current room':str(int(d['current'])),'superseded room':str(int(d['old']))})
for p in [rf'{ZT}(?:最新安排|地點更新):請使用(?P<current>\d+)室,(?:早前通告|舊告示)所列的(?P<old>\d+)室(?:安排)?已(?:取消|不適用)',rf'{ZT}改在(?P<current>\d+)室舉行。舊通告所列的(?P<old>\d+)室已不適用',rf'{ZT}移至(?P<current>\d+)室。請勿依照舊告示前往(?P<old>\d+)室']:
    add('zh','room correction',p,lambda d:{'current room':str(int(d['current'])),'superseded room':str(int(d['old']))})
for lang in RULES:
    cur,old=datepat('cur',lang),datepat('old',lang)
    if lang=='en':
        patterns=[('opening correction',rf'{ET} update:\s*disregard the previous opening date of {old}\.\s*The revised date is {cur}'),('appointment correction',rf'Correction for the {ET}:\s*your appointment is on {cur}\.\s*Please disregard the previous email showing {old}')]
    else:
        patterns=[('opening correction',rf'{ZT}開放日期已改為{cur},原定{old}的安排已取消'),('opening correction',rf'{ZT}將於{cur}開放。早前公布的{old}已不適用'),('appointment correction',rf'{ZT}的預約日期為{cur}。請以本電郵為準,早前所列的{old}已不適用'),('appointment correction',rf'早前電郵所列的{old}已取消。你的{ZT}預約改於{cur}')]
    for kind,p in patterns:
        add(lang,kind,p,lambda d,l=lang:{'current date':dateval(d,'cur',l),'superseded date':dateval(d,'old',l)})

# General and exceptional audience times remain separately keyed.
AUD_EN={'Performers':'performers','Exhibitors':'exhibitors','Invited speakers':'invited speakers','Groups with bookings':'booked groups'}
AUD_ZH={'表演者':'performers','參展者':'exhibitors','獲邀講者':'invited speakers','已預約團體':'booked groups'}
add('en','audience schedule',rf'{ET} schedule\s+General (?P<action>arrival|admission):\s*{ct("general","en")}\s+\*\s*(?P<who>Performers|Exhibitors|Invited speakers|Groups with bookings) (?P<verb>report|arrive|enter) at {ct("special","en")} instead',lambda d:{'general action':'admission' if d['action'].lower()=='admission' else 'arrival','general time':timeval(d,'general'),'exception audience':next(v for k,v in AUD_EN.items() if k.lower()==d['who'].lower()),'exception action':{'enter':'admission','report':'report','arrive':'arrival'}[d['verb'].lower()],'exception time':timeval(d,'special')})
for p in [rf'{ZT}時間表\s+一般(?P<action>到場時間|入場):{ct("general","zh")}\s+\*\s*(?P<who>{"|".join(AUD_ZH)})改於{ct("special","zh")}(?P<verb>報到|到場|入場)',rf'{ZT}一般(?P<action>到場|入場)時間為{ct("general","zh")}。(?P<who>{"|".join(AUD_ZH)})須按附註於{ct("special","zh")}(?P<verb>報到|到場|入場)']:
    add('zh','audience schedule',p,lambda d:{'general action':'admission' if d['action']=='入場' else 'arrival','general time':timeval(d,'general'),'exception audience':AUD_ZH[d['who']],'exception action':{'入場':'admission','報到':'report','到場':'arrival'}[d['verb']],'exception time':timeval(d,'special')})

# Simple time statements: require a complete sentence, not mere matching digits.
for kind,pattern in [('start time',rf'(?:The )?{ET} starts at {ct("","en")}'),('opening time',rf'The {ET} opens at {ct("","en")}'),('appointment time',rf'Your appointment for the {ET} is at {ct("","en")}')]:
    add('en',kind,pattern,lambda d:{'time':timeval(d)})
for kind,patterns in [('start time',[rf'{ZT}於{ct("","zh")}開始',rf'{ct("","zh")}開始{ZT}',rf'{ZT}開始時間:{ct("","zh")}']),('opening time',[rf'{ZT}於{ct("","zh")}開放',rf'{ct("","zh")}起可使用{ZT}']),('appointment time',[rf'你的{ZT}預約時間為{ct("","zh")}',rf'你已預約於{ct("","zh")}辦理{ZT}'])]:
    for p in patterns: add('zh',kind,p,lambda d:{'time':timeval(d)})

def parse_notice(text,lang):
    s=norm(text)
    for kind,pattern,convert in RULES[lang]:
        m=pattern.fullmatch(s)
        if not m: continue
        d=m.groupdict(); topic=d['topic'].strip()
        if lang=='zh' and kind=='door exception':
            if d['cond']=='除非' and '否則' not in s: continue
            if d['cond']=='即使' and '仍' not in s: continue
        # Reject embedded instructions or conditions hidden inside a title slot.
        if lang=='en' and re.search(r'\b(?:can|may|need|where|who|that|must|unless|only|cannot|without|before|after|should|shall|not|bring|pay|submit|register|attend|provided|except|including)\b',topic,re.I): continue
        if lang=='zh' and re.search(r'須|需|可|除非|否則|不得|不准|僅限|只限|即使|請勿|並|且|攜帶|繳付|提交|報名|之後|之前',topic): continue
        try: facts=convert(d)
        except (ValueError,KeyError): continue
        if kind in ('badge order','account order') and d.get('a')==d.get('b') and 'a' in d: continue
        return Notice(kind,topic,facts,text)
    return None

def structured_comparison(chinese,english):
    z,e=parse_notice(chinese,'zh'),parse_notice(english,'en')
    if not z or not e or z.kind!=e.kind: return None
    evidence=[]
    for key in sorted(set(z.facts)|set(e.facts)):
        zv,ev=z.facts.get(key,'not specified'),e.facts.get(key,'not specified')
        if zv!=ev:
            evidence.append(dict(category=key,chinese_value=zv,english_value=ev,chinese_span=z.text,english_span=e.text))
    return z,e,evidence

"""Install V8 over a private frozen namespace, leaving original functions intact."""
from functools import lru_cache
_frozen_notice_check_v8 = check_notice

def _contextual_fallback_v8(chinese,english):
    out=_frozen_notice_check_v8(chinese,english)
    # The inherited date-role heuristic also covers replies and RSVPs. Use a
    # neutral label rather than asserting every deadline is an application.
    out['why']=out['why'].str.replace('application deadline differs:', 'deadline differs:', regex=False)
    return out

@lru_cache(maxsize=512)
def topic_scores_v8(zh,en):
    # Full propositions for NLI, not sentence fragments. Fixed V7 thresholds.
    return (similarity(zh,en), contradiction('本通知涉及'+zh+'。','This notice concerns the '+en+'.'))

def check_notice(chinese_text,english_text):
    parsed=structured_comparison(chinese_text,english_text)
    if parsed is None:
        out=_contextual_fallback_v8(chinese_text,english_text)
        out['decision_path']='frozen_fallback'
        return out
    z,e,evidence=parsed
    sim,contra=topic_scores_v8(z.topic,e.topic)
    # A supported structure does not establish equivalence of the event itself.
    reasons=[]
    subject_uncertain = sim<SIM_T or contra>NLI_T
    # Numeric identifiers in a title must survive semantic matching.
    if set(re.findall(r'\d+',z.topic)) != set(re.findall(r'\d+',e.topic)):
        evidence.append(dict(category='subject identifier',chinese_value=z.topic,english_value=e.topic,chinese_span=z.topic,english_span=e.topic))
    if subject_uncertain and not evidence:
        # Noun-phrase scores are not calibrated like complete notices. Do not
        # create a new alarm solely by removing context. Defer unresolved title
        # equivalence to the existing full-notice checker, preserving its limits.
        out=_contextual_fallback_v8(chinese_text,english_text)
        out['decision_path']='uncertain_subject_fallback'
        out['topic_similarity']=sim
        out['topic_contradiction']=contra
        return out
    for x in evidence:
        reasons.append(f"{x['category']} differs: English {x['english_value']}; Chinese {x['chinese_value']}. Evidence — English [{x['english_span']}]; Chinese [{x['chinese_span']}]")
    return pd.DataFrame([dict(status='⚠️ Check' if reasons else '✅ OK',chinese=chinese_text,english=english_text,why='; '.join(reasons),semantic_evidence=evidence,decision_path='structured_notice',notice_kind=z.kind,topic_similarity=sim,topic_contradiction=contra,english_facts=e.facts,chinese_facts=z.facts)])

CHECKER_VERSION='v8-structured-notice-candidate'

### V9: clause facts
V8's whole-notice patterns only help when the **entire** notice fits one pattern. V9 works one clause at a
time instead: each clause that matches one of our clause patterns becomes a fact (for example
*return postage payer = applicant*, *children age eligibility = 6 to 11*). Facts that appear once in both
versions are compared value by value, and a difference is reported with the exact words from each language.

Everything that is **not** read as a clause fact (and any fact that appears in only one version, or more than
once) is passed to the V8 checker, so it is still checked by the rules and models.

In [ ]:
"""Compositional bilingual clause facts, with unresolved text retained for V8."""
import re
import unicodedata
from collections import defaultdict

CLAUSE_RULES = []
def clause(key, en, zh, en_value=None, zh_value=None):
    CLAUSE_RULES.append((key,re.compile(en,re.I),re.compile(zh),en_value,zh_value))

def normal(text):
    return unicodedata.normalize('NFKC',text).replace('，',',').replace('；',';')

def fixed(key,en,zh):
    clause(key,en,zh,lambda m:'present',lambda m:'present')

fixed('unregistered viewing',r'You may view the records without registering\.',r'查閱紀錄無須登記;')
clause('registration scope',r'Registration is compulsory (only if you want to download a copy|whether or not you want to download a copy)\.',r'(僅在需要下載副本時才|無論是否需要下載副本,均)必須登記。',lambda m:'download only' if m[1].startswith('only') else 'all viewers',lambda m:'download only' if m[1].startswith('僅') else 'all viewers')
fixed('application fee refund',r'Application fees can be refunded\.',r'申請費可退回。')
clause('courier refund on withdrawal before review',r'Courier charges are (excluded|also refunded), even when the application is withdrawn before review\.',r'即使申請在審閱前撤回,亦(不退還|會退還)速遞費。',lambda m:'excluded' if m[1]=='excluded' else 'refunded',lambda m:'excluded' if m[1]=='不退還' else 'refunded')
clause('children age eligibility',r'The desk serves children (aged (\d+) or younger|under (\d+)) for this activity\.',r'活動櫃位只受理((\d+)歲或以下|未滿(\d+)歲)兒童的申請。',lambda m:('<=' if m[2] else '<')+(m[2] or m[3]),lambda m:('<=' if m[2] else '<')+(m[2] or m[3]))
fixed('older children route',r'Older children should use the main booking page\.',r'年齡較大的兒童請使用主預約頁面。')
fixed('prefer upload',r'Use the upload page whenever possible\.',r'請盡量使用上載頁面;')
clause('postal submission condition',r'Send the file by post only if the page is (unavailable|available)\.',r'只在頁面(無法使用|可以使用)時才可郵寄文件。',lambda m:'page unavailable' if m[1]=='unavailable' else 'page available',lambda m:'page unavailable' if m[1]=='無法使用' else 'page available')
fixed('held for collection',r'The item will be held for collection\.',r'物件會保留待領;')
clause('authorised representative obligation',r'A nominated representative (may|must) collect it with written authorisation\.',r'獲提名的代表持書面授權(可|必須)代領。',lambda m:'permitted' if m[1]=='may' else 'required',lambda m:'permitted' if m[1]=='可' else 'required')
fixed('outbound postage payer',r'The requester pays for delivery\.',r'寄出費用由申請人負責;')
clause('return postage payer',r'For a return, the (recipient|requester) pays the postage\.',r'退回物件的郵費由(收件人|申請人)支付。',lambda m:m[1],lambda m:{'收件人':'recipient','申請人':'requester'}[m[1]])
clause('other items after partial cancellation',r'If you cancel one item, the other items are (kept on the order|removed from the order)\.',r'如取消其中一項,其餘項目(仍保留在訂單內|亦會從訂單移除)。',lambda m:'retained' if m[1].startswith('kept') else 'removed',lambda m:'retained' if m[1].startswith('仍') else 'removed')
fixed('confirmation number retained',r'Your confirmation number stays the same\.',r'確認編號維持不變。')
clause('retention start and duration',r'The retention period starts on the day staff (receive|approve) the form; it lasts (\d+) days\.',r'紀錄由職員(收到|批准)表格當日起保留(\d+)日。',lambda m:('receipt' if m[1]=='receive' else 'approval')+'; '+m[2]+' days',lambda m:('receipt' if m[1]=='收到' else 'approval')+'; '+m[2]+' days')
clause('priority when simultaneous',r'When both groups arrive together, serve people (with|without) an appointment first\.',r'兩組人士同時抵達時,請先為(已|未)預約人士提供服務。',lambda m:'booked' if m[1]=='with' else 'unbooked',lambda m:'booked' if m[1]=='已' else 'unbooked')
fixed('single waiting queue',r'Everyone should wait in the same line\.',r'所有人須在同一隊伍等候。')
clause('fee waiver threshold',r'The fee is waived on orders of (at least|more than) HK\$(\d+)\.',r'訂單金額(達|超過)(\d+)港元可免付費用;',lambda m:('>=' if m[1]=='at least' else '>')+m[2]+' HKD',lambda m:('>=' if m[1]=='達' else '>')+m[2]+' HKD')
fixed('ordinary fee otherwise',r'Otherwise, the usual fee applies\.',r'其餘訂單照常收費。')
clause('automated acknowledgement status',r'An automated reply confirms (receipt|approval) of the request\.',r'系統電郵只確認已(收到|批准)申請。',lambda m:m[1],lambda m:'receipt' if m[1]=='收到' else 'approval')
fixed('later decision after review',r'A decision will be sent separately after staff review\.',r'職員審查後會另行通知結果。')
fixed('countersigning officer',r'The duty officer countersigns each form;',r'每份表格由當值主任加簽,')
clause('signed original custodian',r'the (applicant|duty officer) keeps the signed original\.',r'正本由(申請人|當值主任)保管。',lambda m:m[1],lambda m:'applicant' if m[1]=='申請人' else 'duty officer')
fixed('shared scan destination',r'A scan goes into the shared folder\.',r'掃描本存入共用資料夾。')

# Common concrete attributes and instructions can be composed with any context.
clause('entry gate colour',r'Entry for this activity is through the (green|blue|red|yellow) gate\.',r'本活動參加者請經(綠色|藍色|紅色|黃色)閘口入場;',lambda m:m[1],lambda m:{'綠色':'green','藍色':'blue','紅色':'red','黃色':'yellow'}[m[1]])
fixed('concourse available',r'The main concourse remains open as usual\.',r'大堂照常開放。')
clause('signed record tray',r'When finished, leave the signed record in the (outgoing|incoming) tray beside the printer\.',r'簽署後,請將紀錄放入打印機旁的(待發|來件)文件盤。',lambda m:m[1],lambda m:'outgoing' if m[1]=='待發' else 'incoming')
clause('card reference side',r'The reference is printed on the (back|front) of your card\.',r'編號印在卡的(背面|正面),',lambda m:m[1],lambda m:'back' if m[1]=='背面' else 'front')
fixed('reference spacing',r'Type it without spaces\.',r'輸入時請勿留空格。')
clause('access code delivery channel',r'The updated access code will be sent by (text message|email)\.',r'更新後的通行碼將以(短訊|電郵)發送,',lambda m:m[1],lambda m:'text message' if m[1]=='短訊' else 'email')
fixed('contact details current',r'Please keep your contact details current\.',r'請確保聯絡資料為最新。')

def extract_clauses(text,lang):
    text=normal(text)
    facts=defaultdict(list)
    spans=[]
    for key,ep,zp,ev,zv in CLAUSE_RULES:
        pattern,value=(ep,ev) if lang=='en' else (zp,zv)
        for match in pattern.finditer(text):
            # Only consume whole clause boundaries, including labelled list items.
            prefix=text[:match.start()].rstrip()
            if prefix and prefix[-1] not in '.。,;:!?\n' and not text[:match.start()].endswith('\n'):
                continue
            if any(match.start()<end and match.end()>start for start,end in spans):
                continue
            facts[key].append(dict(value=value(match),span=match[0]))
            spans.append(match.span())
    chars=list(text)
    for start,end in spans:
        chars[start:end]=' '* (end-start)
    remainder=''.join(chars)
    remainder=re.sub(r'(?m)^\s*[-─]+\s*$','',remainder)
    return facts,remainder.strip()

_v8_original_check=check_notice

def check_notice(chinese_text,english_text):
    z,zremain=extract_clauses(chinese_text,'zh')
    e,eremain=extract_clauses(english_text,'en')
    shared=set(z)&set(e)
    if not shared:
        return _v8_original_check(chinese_text,english_text)
    # Put unmatched or repeated facts back into residual text for contextual review.
    evidence=[]
    for key in sorted(set(z)|set(e)):
        if key not in shared or len(z[key])!=1 or len(e[key])!=1:
            zremain+='\n'+'\n'.join(x['span'] for x in z[key])
            eremain+='\n'+'\n'.join(x['span'] for x in e[key])
            continue
        if z[key][0]['value']!=e[key][0]['value']:
            evidence.append(dict(category=key,chinese_value=z[key][0]['value'],english_value=e[key][0]['value'],chinese_span=z[key][0]['span'],english_span=e[key][0]['span']))
    if zremain.strip() or eremain.strip():
        result=_v8_original_check(zremain,eremain)
        result['decision_path']='clause_facts_with_residual_review'
    else:
        result=pd.DataFrame([dict(status='✅ OK',chinese=chinese_text,english=english_text,why='',decision_path='clause_facts')])
    if evidence:
        reasons=[f"{x['category']} differs: English {x['english_value']} [{x['english_span']}]; Chinese {x['chinese_value']} [{x['chinese_span']}]" for x in evidence]
        result=pd.concat([result,pd.DataFrame([dict(status='⚠️ Check',chinese=chinese_text,english=english_text,why='; '.join(reasons),semantic_evidence=evidence,decision_path='clause_facts')])],ignore_index=True)
    return result

V8_check_notice = _v8_original_check    # kept for the V8 vs V9 comparison in section 11
CHECKER_VERSION = "V9"
print("Checker:", CHECKER_VERSION, "| thresholds:", SIM_T, NLI_T, "|", len(CLAUSE_RULES), "clause patterns")

### Guard cases
Small hand-made pairs we used while writing the V8 rules (development checks, not a test score). Half are
correct translations that must **not** be flagged; half change one detail and **must** be flagged, to make
sure no rule hides a real error.

In [ ]:
GUARDS = [
 {
  "case_id": "test_clean_normalisation_1",
  "chinese": "運動會兩年一度舉行。",
  "english": "The games are held biennially.",
  "expected_status": "clean",
  "split": "development_guard"
 },
 {
  "case_id": "test_clean_normalisation_2",
  "chinese": "運動會每兩年舉行。",
  "english": "The games take place every other year.",
  "expected_status": "clean",
  "split": "development_guard"
 },
 {
  "case_id": "test_clean_normalisation_3",
  "chinese": "運動會一年一度舉行。",
  "english": "The games are held annually.",
  "expected_status": "clean",
  "split": "development_guard"
 },
 {
  "case_id": "test_clean_normalisation_4",
  "chinese": "學生可獲半費優惠。",
  "english": "Students receive a 50 per cent concession.",
  "expected_status": "clean",
  "split": "development_guard"
 },
 {
  "case_id": "test_clean_normalisation_5",
  "chinese": "學生門票半價。",
  "english": "Student tickets are half price.",
  "expected_status": "clean",
  "split": "development_guard"
 },
 {
  "case_id": "test_clean_normalisation_6",
  "chinese": "評審將於第三季完成。",
  "english": "Assessment will finish in the third quarter.",
  "expected_status": "clean",
  "split": "development_guard"
 },
 {
  "case_id": "test_clean_normalisation_7",
  "chinese": "設有五人足球。",
  "english": "Futsal is available.",
  "expected_status": "clean",
  "split": "development_guard"
 },
 {
  "case_id": "test_clean_normalisation_8",
  "chinese": "各項目分成兩個組別。",
  "english": "All events are divided into two groups.",
  "expected_status": "clean",
  "split": "development_guard"
 },
 {
  "case_id": "test_clean_normalisation_9",
  "chinese": "請核對識別編號。",
  "english": "Please check the identification number.",
  "expected_status": "clean",
  "split": "development_guard"
 },
 {
  "case_id": "test_clean_normalisation_10",
  "chinese": "詳情請瀏覽hk.art.museum/tc/web/ma/test.html。",
  "english": "For details visit hk.art.museum/en/web/ma/test.html.",
  "expected_status": "clean",
  "split": "development_guard"
 },
 {
  "case_id": "test_clean_normalisation_11",
  "chinese": "請瀏覽https://example.org/tc/form?id=12。",
  "english": "Please visit https://example.org/en/form?id=12.",
  "expected_status": "clean",
  "split": "development_guard"
 },
 {
  "case_id": "test_suppression_must_not_hide_errors_1",
  "chinese": "運動會兩年一度舉行。",
  "english": "The games are held annually.",
  "expected_status": "issue",
  "split": "development_guard"
 },
 {
  "case_id": "test_suppression_must_not_hide_errors_2",
  "chinese": "運動會每三年舉行。",
  "english": "The games take place every other year.",
  "expected_status": "issue",
  "split": "development_guard"
 },
 {
  "case_id": "test_suppression_must_not_hide_errors_3",
  "chinese": "學生可獲半費優惠。",
  "english": "Students receive a 25 per cent concession.",
  "expected_status": "issue",
  "split": "development_guard"
 },
 {
  "case_id": "test_suppression_must_not_hide_errors_4",
  "chinese": "評審將於第三季完成。",
  "english": "Assessment will finish in the fourth quarter.",
  "expected_status": "issue",
  "split": "development_guard"
 },
 {
  "case_id": "test_suppression_must_not_hide_errors_5",
  "chinese": "設有七人足球。",
  "english": "Futsal is available.",
  "expected_status": "issue",
  "split": "development_guard"
 },
 {
  "case_id": "test_suppression_must_not_hide_errors_6",
  "chinese": "各項目不分組別。",
  "english": "All events are divided into groups.",
  "expected_status": "issue",
  "split": "development_guard"
 },
 {
  "case_id": "test_suppression_must_not_hide_errors_7",
  "chinese": "請瀏覽https://example.org/tc/form?id=12。",
  "english": "Please visit https://example.org/en/form?id=21.",
  "expected_status": "issue",
  "split": "development_guard"
 },
 {
  "case_id": "test_suppression_must_not_hide_errors_8",
  "chinese": "請瀏覽https://example.org/tc/form。",
  "english": "Please visit https://different.org/en/form.",
  "expected_status": "issue",
  "split": "development_guard"
 },
 {
  "case_id": "test_suppression_must_not_hide_errors_9",
  "chinese": "請瀏覽https://example.org/tc/refund。",
  "english": "Please visit https://example.org/en/payment.",
  "expected_status": "issue",
  "split": "development_guard"
 },
 {
  "case_id": "test_clean_relations_1",
  "chinese": "如住址證明以家長名義發出,另須上載證明親屬關係的文件。",
  "english": "If the proof is in your parent's name, also upload a document showing your relationship.",
  "expected_status": "clean",
  "split": "development_guard"
 },
 {
  "case_id": "test_clean_relations_2",
  "chinese": "如住址證明由家長簽發,另須上載證明親屬關係的文件。",
  "english": "If the proof is issued by your parent, also upload a document showing your relationship.",
  "expected_status": "clean",
  "split": "development_guard"
 },
 {
  "case_id": "test_clean_relations_3",
  "chinese": "代領人須帶同收件人的身分證明文件副本及代領人本人的身分證明文件正本。",
  "english": "The proxy must bring a copy of your ID and their own original ID.",
  "expected_status": "clean",
  "split": "development_guard"
 },
 {
  "case_id": "test_clean_relations_4",
  "chinese": "代領人須帶同收件人的身分證明文件正本及代領人本人的身分證明文件副本。",
  "english": "The proxy must bring your original ID and a copy of their own ID.",
  "expected_status": "clean",
  "split": "development_guard"
 },
 {
  "case_id": "test_clean_relations_5",
  "chinese": "如未能親身取件,可委託他人代領。",
  "english": "You may send a proxy if you cannot collect your parcel.",
  "expected_status": "clean",
  "split": "development_guard"
 },
 {
  "case_id": "test_clean_relations_6",
  "chinese": "如未能在原定時段親身取件,可委託他人代領。",
  "english": "You may send a proxy if you cannot collect your parcel at the original slot.",
  "expected_status": "clean",
  "split": "development_guard"
 },
 {
  "case_id": "test_clean_relations_7",
  "chinese": "只有在兩名同行者均為其照顧者的情況下才可預留。",
  "english": "You may reserve only if both companions are your carers.",
  "expected_status": "clean",
  "split": "development_guard"
 },
 {
  "case_id": "test_clean_relations_8",
  "chinese": "其他合資格人士仍可報名。",
  "english": "Other eligible applicants can still register.",
  "expected_status": "clean",
  "split": "development_guard"
 },
 {
  "case_id": "test_clean_relations_9",
  "chinese": "其他合資格人士須待優先階段結束後才可報名。",
  "english": "Other eligible applicants may register after the priority period ends.",
  "expected_status": "clean",
  "split": "development_guard"
 },
 {
  "case_id": "test_changed_relations_1",
  "chinese": "如住址證明由家長簽發,另須上載證明親屬關係的文件。",
  "english": "If the proof is in your parent's name, also upload a document showing your relationship.",
  "expected_status": "issue",
  "split": "development_guard"
 },
 {
  "case_id": "test_changed_relations_2",
  "chinese": "代領人須帶同收件人的身分證明文件正本及代領人本人的身分證明文件副本。",
  "english": "The proxy must bring a copy of your ID and their own original ID.",
  "expected_status": "issue",
  "split": "development_guard"
 },
 {
  "case_id": "test_changed_relations_3",
  "chinese": "如未能在原定時段親身取件,可委託他人代領。",
  "english": "You may send a proxy if you cannot collect your parcel.",
  "expected_status": "issue",
  "split": "development_guard"
 },
 {
  "case_id": "test_changed_relations_4",
  "chinese": "只有在其中一名同行者為其照顧者的情況下才可預留。",
  "english": "You may reserve only if both companions are your carers.",
  "expected_status": "issue",
  "split": "development_guard"
 },
 {
  "case_id": "test_changed_relations_5",
  "chinese": "其他合資格人士須待優先階段結束後才可報名。",
  "english": "Other eligible applicants can still register.",
  "expected_status": "issue",
  "split": "development_guard"
 }
]

def notice_flagged(result):
    # a notice counts as flagged if any row gives a reason
    return result["why"].fillna("").astype(str).str.strip().ne("").any()

guard_rows = []
for g in GUARDS:
    flagged = notice_flagged(check_notice(g["chinese"], g["english"]))
    guard_rows.append({"expected": g["expected_status"], "flagged": flagged,
                       "pass": flagged == (g["expected_status"] == "issue"),
                       "chinese": g["chinese"], "english": g["english"]})
guard_rows = pd.DataFrame(guard_rows)
print(f"Guard cases passed: {guard_rows['pass'].sum()} / {len(guard_rows)}")
display(guard_rows[~guard_rows["pass"]])

## 6. Score every pair once
The two model scores are computed once per row, then the locked settings are used:
- `SIM_T = 0.40`: firm warning if LaBSE similarity is **below** this
- `NLI_T = 0.65`: firm warning if the contradiction probability is **above** this
- `NLI_LOOK = 0.30`: a softer "worth a look" mark between 0.30 and 0.65 (from v6)

These were chosen on development data in v3 and v4 and have not changed since.

In [ ]:
df["similarity"] = [similarity(z, e) for z, e in zip(df.chinese, df.english)]
df["contradiction"] = [contradiction(z, e) for z, e in zip(df.chinese, df.english)]
df["rule_reasons"] = [rule_reasons(z, e) for z, e in zip(df.chinese, df.english)]
SIM_T_B, NLI_T_B = SIM_T, NLI_T     # B (models only) uses the same thresholds
NLI_LOOK = 0.30
df[["split", "error_type", "similarity", "contradiction", "rule_reasons", "english"]].round(2)

### Choosing the contradiction threshold
Our first automatic tuning used only the 12 dev notices. On those, every threshold from 0.30 to 0.95 looked
perfect, so it could not tell them apart: 0.30 gave false alarms on other rows, 0.95 missed real errors.
So we looked at the trade-off across **all** starter rows and chose the threshold that catches the most errors
with no false alarms. Because all starter rows were used for this choice, they are **development data**.
The honest tests are the real notices in section 8 and the held-out set in section 12.

In [ ]:
# Trade-off on all development rows: how the contradiction threshold changes errors caught vs false alarms
err = df.has_error.values == 1
rows_t = []
for t in [0.3, 0.4, 0.5, 0.6, 0.65, 0.7, 0.8, 0.9, 0.95]:
    flags = np.array([bool(r) or c > t or s < SIM_T
                      for r, c, s in zip(df.rule_reasons, df.contradiction, df.similarity)])
    rows_t.append({"contradiction threshold": t,
                   "errors caught": f"{(flags & err).sum()} / {err.sum()}",
                   "false alarms": f"{(flags & ~err).sum()} / {(~err).sum()}"})
pd.DataFrame(rows_t)

## 7. Development results
Six systems on the same rows:
- **Baseline:** length rule
- **A. Rules only:** numbers, negation, date roles, weekdays (no AI)
- **B. Pretrained models only:** LaBSE + mDeBERTa with thresholds, none of our rules
- **C. V8 sentence checker:** models + rules + V8 evidence rules
- **D. C + worth a look:** C, plus a softer 🔎 mark when the contradiction score is between 0.30 and 0.65
- **E. V9 full checker:** clause facts, then C for everything the clause layer does not read. **This is V9.**

We report **errors caught** and **false alarms** separately, then break down by error type.
These rows helped choose our settings, so treat these numbers as a development check, not the final score.

In [ ]:
def look_flag(r):
    # C's warnings, plus a softer "worth a look" mark when the contradiction score is between the two thresholds
    flagged, reasons = checker_flag(r.chinese, r.english, r.similarity, r.contradiction)
    if not flagged and r.contradiction > NLI_LOOK:
        return True, [f"worth a look: the two versions might disagree ({r.contradiction:.0%})"]
    return flagged, reasons

def v9_flag(r):
    result = check_notice(r.chinese, r.english)
    reasons = [w for w in result["why"].fillna("").astype(str) if w.strip()]
    return bool(reasons), reasons

SYSTEMS = {
    "Baseline: length rule": lambda r: baseline_flag(r.chinese, r.english),
    "A. Rules only": lambda r: (bool(r.rule_reasons), r.rule_reasons),
    "B. Pretrained models only": lambda r: (lambda rs: (bool(rs), rs))(model_reasons(r.similarity, r.contradiction, SIM_T_B, NLI_T_B)),
    "C. V8 sentence checker": lambda r: checker_flag(r.chinese, r.english, r.similarity, r.contradiction),
    "D. C + worth a look": look_flag,
    "E. V9 full checker": v9_flag,
}
FULL = "E. V9 full checker"
def evaluate(frame, title):
    frame = frame.copy()
    summary = []
    for name, fn in SYSTEMS.items():
        out = frame.apply(fn, axis=1)
        frame[name] = [o[0] for o in out]
        frame[name + " reasons"] = [o[1] for o in out]
        errs, oks = frame[frame.has_error == 1], frame[frame.has_error == 0]
        summary.append({"system": name,
                        "errors caught": f"{int(errs[name].sum())} / {len(errs)}",
                        "false alarms": f"{int(oks[name].sum())} / {len(oks)}"})
    print(title)
    display(pd.DataFrame(summary))
    print("Errors caught, by error type")
    display(frame[frame.has_error == 1].groupby("error_type")[list(SYSTEMS)].agg(lambda s: f"{int(s.sum())} / {len(s)}"))
    print("MISSED ERRORS (full checker)")
    display(frame[(frame.has_error == 1) & (~frame[FULL])][["error_type", "chinese", "english", "similarity", "contradiction"]])
    print("FALSE ALARMS (full checker)")
    display(frame[(frame.has_error == 0) & (frame[FULL])][["chinese", "english", FULL + " reasons"]])
    return frame

dev_results = evaluate(df, "DEVELOPMENT RESULTS (all starter rows)")

## 8. Tests on real notices
Both files hold real Hong Kong government notices. Each was scored **blind** once, and those blind scores are
what we report:
- `fresh_test.csv` (first blind test, notebook v3): **20 of 22 errors caught, 12 false alarms on 26 correct sentences.**
- `fresh_test_2.csv` (final blind test, notebook v4): **19 of 21 errors caught, 7 false alarms on 26 correct sentences.**

V5 to V9 were all written **after** those runs, so the tables below are *not blind*: they show how the
later versions do on notices we have already seen.

In [ ]:
TEST_FILES = {"fresh_test.csv": "FIRST REAL-NOTICE SET, NOT BLIND (blind score was 20/22 caught, 12/26 false alarms)",
              "fresh_test_2.csv": "FINAL REAL-NOTICE SET, NOT BLIND (blind score was 19/21 caught, 7/26 false alarms)"}
test_results = {}
for path, title in TEST_FILES.items():
    if not os.path.exists(path):
        print(f"No {path} found. Upload it, then run this cell.")
        continue
    fresh = pd.read_csv(path)
    fresh["similarity"] = [similarity(z, e) for z, e in zip(fresh.chinese, fresh.english)]
    fresh["contradiction"] = [contradiction(z, e) for z, e in zip(fresh.chinese, fresh.english)]
    fresh["rule_reasons"] = [rule_reasons(z, e) for z, e in zip(fresh.chinese, fresh.english)]
    print(len(fresh), "rows:", fresh.has_error.value_counts().to_dict())
    test_results[path] = evaluate(fresh, title)

## 9. Check a whole notice
`check_notice(chinese_text, english_text)` is the V9 checker. Clause facts are compared first. The rest of
the notice is split into sentences, Chinese and English sentences are matched **one-to-one** with LaBSE (so
reordered paragraphs still match), and every pair goes through the rules and models. Sentences with no
match are flagged, and deadline vs event date is also compared across the whole notice.

### Three test notices
1. **Clean pair:** a correct translation. Should be all ✅.
2. **Swapped dates:** both versions contain 20 and 27 November, but the deadline and event date are swapped in English. Should flag the deadline.
3. **Reordered and reformatted, still correct:** same meaning, sentences in a different order, date written differently. Should stay ✅ (this shows restraint, not just detection).

In [ ]:
fixture_zh = "報名截止日期為11月20日。活動將於11月27日舉行。參加者毋須繳付任何費用。"

fixtures = {
    "1. Clean pair (expect all OK)":
        "The application deadline is 20 November. The activity will be held on 27 November. Participants do not need to pay any fee.",
    "2. Swapped dates (expect a deadline warning)":
        "The application deadline is 27 November. The activity will be held on 20 November. Participants do not need to pay any fee.",
    "3. Reordered and reformatted (expect all OK)":
        "Participants do not need to pay any fee. The activity will be held on November 27. The application deadline is November 20.",
}
pd.set_option("display.max_colwidth", 120)
for name, english in fixtures.items():
    print("\n" + name)
    display(check_notice(fixture_zh, english)[["status", "chinese", "english", "why"]])

## 10. Demo app

In [ ]:
import gradio as gr

def demo_check(chinese_text, english_text):
    return check_notice(chinese_text, english_text)[["status", "chinese", "english", "why"]]

gr.Interface(
    fn=demo_check,
    inputs=[gr.Textbox(label="Traditional Chinese version", lines=8, value=fixture_zh),
            gr.Textbox(label="English version", lines=8, value=fixtures["2. Swapped dates (expect a deadline warning)"])],
    outputs=gr.Dataframe(label="Findings", wrap=True),
    title="Public Notice Check",
    description="A last check before a bilingual notice goes out. It flags sentences to review; a person makes the final call.",
).launch()

## 11. V8 vs V9 on 1,000 development cases
A synthetic set of 1,000 bilingual notices (300 with a real mismatch, 700 correct), in three difficulty
levels. We used it **while building V9**, so this is a development result, not a test score. When V9 was
first run, it printed: V8 **81.4%** accuracy → V9 **97.9%** (errors caught 248 → 293 of 300, false alarms
134 → 14 of 700). Running it here should give the same numbers.

In [ ]:
import time, json
from collections import Counter

def run_notice_set(path, title):
    cases = pd.read_csv(path, encoding="utf-8-sig")
    counts = {v: Counter() for v in ["V8", "V9"]}
    tiers = {v: {} for v in ["V8", "V9"]}
    rows, start = [], time.time()
    for n, case in enumerate(cases.itertuples(), 1):
        truth = case.expected_status == "issue"
        row = {"case_id": case.case_id, "difficulty": case.difficulty, "expected": case.expected_status}
        for version, checker in [("V8", V8_check_notice), ("V9", check_notice)]:
            flagged = bool(notice_flagged(checker(case.chinese, case.english)))
            outcome = ("T" if flagged == truth else "F") + ("P" if flagged else "N")
            counts[version][outcome] += 1
            tiers[version].setdefault(case.difficulty, Counter())[outcome] += 1
            row[version] = outcome
        rows.append(row)
        if n % 100 == 0:
            print(f"{n}/{len(cases)} notices checked ({time.time() - start:.0f} s)", flush=True)
    summary = []
    for tier in ["routine", "difficult", "edge", "all"]:
        a = counts["V8"] if tier == "all" else tiers["V8"].get(tier, Counter())
        b = counts["V9"] if tier == "all" else tiers["V9"].get(tier, Counter())
        n, pos = sum(a.values()), a["TP"] + a["FN"]
        if not n:
            continue
        summary.append({"difficulty": tier, "cases": n,
                        "accuracy V8 → V9": f"{(a['TP'] + a['TN']) / n:.1%} → {(b['TP'] + b['TN']) / n:.1%}",
                        "errors caught V8 → V9": f"{a['TP']}/{pos} → {b['TP']}/{pos}",
                        "false alarms V8 → V9": f"{a['FP']}/{n - pos} → {b['FP']}/{n - pos}"})
    print(title)
    display(pd.DataFrame(summary))
    out = pd.DataFrame(rows)
    out.to_csv(path.replace(".csv", "_predictions.csv"), index=False)
    return out

if os.path.exists("v9_development_1000.csv"):
    dev_1000 = run_notice_set("v9_development_1000.csv", "DEVELOPMENT SET (used while building V9)")
else:
    print("No v9_development_1000.csv, skipping.")

## 12. Held-out test: 1,000 new synthetic cases
`v9_unseen_1000.csv` was made the same way as the development set, but from **32 different topics** that share
no case or topic with it, and it was **not used while building V9**. We run it **once** and report whatever it
prints, including where V9 does worse.

It is still synthetic: 32 topics, each written about 30 ways. It tests whether V9's rules generalise to new
wording, not how it does on real community-centre notices (section 8 does that).

In [ ]:
if os.path.exists("v9_unseen_1000.csv"):
    unseen_1000 = run_notice_set("v9_unseen_1000.csv", "HELD-OUT SET (not used while building V9), SCORED ONCE")
else:
    print("No v9_unseen_1000.csv, skipping.")

## 13. Our 60-case whole-notice benchmark (optional)
Runs only if `public_notice_benchmark.csv` has been uploaded. It saves what the checker says for every case
to `public_notice_predictions.jsonl`, using the same locked thresholds as everything above.

In [ ]:
import csv, json, time
from pathlib import Path
bench_path = Path("public_notice_benchmark.csv")
if bench_path.exists():
    with open(bench_path, encoding="utf-8-sig", newline="") as source:
        benchmark_cases = list(csv.DictReader(source))
    print(len(benchmark_cases), "cases; thresholds", SIM_T, NLI_T, "device", device)
    assert len(split_zh("網上報到於中午12時截止，不設即場名額。")) == 2
    output_path = Path("public_notice_predictions.jsonl")
    with output_path.open("w", encoding="utf-8") as sink:
        for n, case in enumerate(benchmark_cases, 1):
            start = time.perf_counter()
            result = {"case_id": case["case_id"]}
            try:
                result["rows"] = check_notice(case["chinese"], case["english"]).to_dict("records")
            except Exception as exc:
                result["error"] = repr(exc)
            result["seconds"] = round(time.perf_counter() - start, 3)
            sink.write(json.dumps(result, ensure_ascii=False) + "\n")
            if n % 10 == 0:
                print(n, "/", len(benchmark_cases))
    print("Benchmark complete: see", output_path)
else:
    print("No public_notice_benchmark.csv uploaded, skipping.")

## 14. Sources
- LaBSE: Feng et al., *Language-agnostic BERT Sentence Embedding* · https://huggingface.co/sentence-transformers/LaBSE
- mDeBERTa NLI: Laurer et al. · https://huggingface.co/MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7 (MIT licence)
- sentence-transformers, transformers, PyTorch, pandas, NumPy, SciPy, Gradio, Google Colab
- Real notices: Hong Kong Government press releases, https://www.info.gov.hk/gia/general/ (links in the `source` column of the fresh test files)
- Development notices (`notices.csv`) and the two 1,000-case sets are synthetic
- AI assistance: Claude (Anthropic) and Gemini (Google) were used while building this project, as the hackathon rules allow